# 01 - Loading and joining the data

In this notebook I load the two Kaggle train files, join them into one table and check the basic numbers: rows, columns, fraud rate and time range.

## Data notes (from the Kaggle data page)

The data is broken into two files `identity` and `transaction`, which are joined by `TransactionID`. Not all transactions have corresponding identity information.
Categorical Features - Transaction

* `ProductCD`
* `card1` - `card6`
* `addr1`, `addr2`
* `P_emaildomain`
* `R_emaildomain`
* `M1` - `M9`

Categorical Features - Identity

* `DeviceType`
* `DeviceInfo`
* `id_12` - `id_38`

The `TransactionDT` feature is a timedelta from a given reference datetime (not an actual timestamp).
You can read more about the data from [this post by the competition host](https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203).
Files

* train_{transaction, identity}.csv - the training set
* test_{transaction, identity}.csv - the test set (you must predict the `isFraud` value for these observations)
* sample_submission.csv - a sample submission file in the correct format

## 1. Transaction file

This is the main file. Each row is one payment: amount, card, product, email domain and `isFraud`, which is the answer the model learns from.

In [5]:
# pandas for loading and working with the tables
import pandas as pd

# path goes up one folder from notebooks/ to reach data/raw/
df = pd.read_csv("../data/raw/train_transaction.csv")
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# (rows, columns) of the transaction table
print(df.shape)

(590540, 394)


**Result:** 590,540 transactions and 394 columns.

## 2. Identity file

This file has device information for each payment: device type, operating system, browser, screen size. I use the **train** file because only the train data has `isFraud` labels.

In [11]:
import pandas as pd

# train identity file, not test: test has no isFraud labels and its IDs would not match
df1 = pd.read_csv("../data/raw/train_identity.csv")
df1.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [13]:
# (rows, columns) of the identity table
df1.shape

(144233, 41)

**Result:** 144,233 rows and 41 columns. So only about 24% of transactions have device data.

## 3. Joining the two tables

The model needs one table where each row is one transaction with everything known about it.

- **Merge, not concat:** concat lines rows up by position, but the two files are not in the same order (identity row 0 is transaction 2987004, which is row 4 in the transaction file). Merge matches rows by `TransactionID`.
- **Left join, not inner:** an inner join keeps only transactions that have device data and would delete about 76% of the data. A left join keeps every transaction, and the device columns are empty (NaN) where there is no identity row. In real life the model has to score payments without device data too.

In [15]:
# left join on TransactionID: keeps every transaction, device columns are NaN where there is no identity row
df_merged = pd.merge(df, df1, on='TransactionID', how='left')
df_merged.shape

(590540, 434)

**Result:** 590,540 rows (no transaction lost) and 434 columns (394 + 41, minus 1 because `TransactionID` appears only once).

In [16]:
# row labels pandas adds (0, 1, 2 ...), not the transaction ID
df_merged.index


RangeIndex(start=0, stop=590540, step=1)

In [17]:
# quick look at the merged table
df_merged


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Fraud rate

`isFraud` is 0 for a genuine payment and 1 for fraud. Dividing the count of each value by the total number of rows gives the share of each.

In [25]:
# share of genuine (0) and fraud (1): count of each value divided by the total number of rows
df_merged['isFraud'].value_counts()/len(df_merged)

isFraud
0    0.96501
1    0.03499
Name: count, dtype: float64

**Result:** 569,877 genuine (96.5%) and 20,663 fraud (3.5%). Fraud is about 1 in every 29 transactions, so the data is heavily imbalanced.

**Why accuracy is not enough:** a model that says "not fraud" for every transaction would get 96.5% accuracy and still catch zero fraud. So I will measure recall (how much of the real fraud is caught), precision (how many alerts are really fraud) and PR-AUC, and use class weights so the model does not ignore the rare fraud cases.

## 5. Time range

`TransactionDT` is not a real date. It is the number of seconds since a hidden reference point. Dividing by 86,400 (seconds in a day) turns it into days.

In [32]:
# TransactionDT is in seconds from a hidden start point; 86400 seconds = 1 day
print(df_merged['TransactionDT'].min()/86400)
print(df_merged['TransactionDT'].max()/86400)

1.0
182.99920138888888


**Result:** the data starts on day 1 and ends on day 183, so it covers about 182 days, roughly 6 months.

**Why this matters for the split:** in real life the model is trained on past payments and has to catch fraud in future ones. So I will split by time: the earliest months for training, the next part for validation and the last part for testing.

A random split would be cheating. Fraudsters often make many payments with the same card in a short time. With a random split, some of those payments end up in training and the rest in testing, so the model can simply remember the card instead of learning what fraud looks like. The score looks great but would not hold up on new fraudsters next month. A time split gives a lower but honest score.

## 6. Duplicates

Each payment should appear only once. If a transaction were repeated, the model would learn from it twice and the fraud counts would be wrong.

In [39]:
# True if this TransactionID was already seen above; counting True/False shows how many duplicates there are
print(df_merged['TransactionID'].duplicated().sum())

0


**Result:** all 590,540 values are False, so there are no duplicate transactions. Every `TransactionID` is unique, and the left join did not create extra rows.

## Day 1 summary

| Check | Result |
|---|---|
| Transaction file | 590,540 rows, 394 columns |
| Identity file | 144,233 rows, 41 columns (about 24% of transactions have device data) |
| Merged (left join) | 590,540 rows, 434 columns |
| Fraud rate | 3.5% (20,663 fraud) |
| Time range | day 1 to day 183, about 6 months |
| Duplicates | 0 |

**Next:** missing values and data quality.

# Day 2 - Data quality

## 7. Missing values

Many columns have empty values (NaN). I check how much is missing in each column, first as a count and then as a percentage.

In [41]:
# number of empty (NaN) values in each column
df_merged.isnull().sum()

TransactionID          0
isFraud                0
TransactionDT          0
TransactionAmt         0
ProductCD              0
                   ...  
id_36             449555
id_37             449555
id_38             449555
DeviceType        449730
DeviceInfo        471874
Length: 434, dtype: int64

**Result:** the key columns (`TransactionID`, `isFraud`, `TransactionDT`, `TransactionAmt`, `ProductCD`) have no missing values. `DeviceType` and `DeviceInfo` are missing for about 450,000-470,000 rows, which matches the transactions with no identity row after the left join.

In [43]:
# share of empty values in each column, as a percentage (average of True/False x 100)
df_merged.isnull().mean()*100

TransactionID      0.000000
isFraud            0.000000
TransactionDT      0.000000
TransactionAmt     0.000000
ProductCD          0.000000
                    ...    
id_36             76.126088
id_37             76.126088
id_38             76.126088
DeviceType        76.155722
DeviceInfo        79.905510
Length: 434, dtype: float64

**Result:** `id_36` to `id_38` are 76.1% empty, exactly the share of transactions with no identity data (100% - 24%). `DeviceInfo` is 79.9% empty, so some transactions have an identity row but still no device name.

- First line: stores the missing percentage for every column.
- Second line: shows the 10 columns with the highest missing percentages.
ascending=False means largest to smallest.

In [45]:
# missing % per column, saved so I can sort and filter it
missing_pct = df_merged.isnull().mean() * 100

# 10 emptiest columns first
missing_pct.sort_values(ascending=False).head(10)

id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
dtype: float64

**Result:** the emptiest columns are identity columns like `id_24`, `id_25`, `id_07` and `id_08`, which are about 99% empty. `dist2` is the emptiest transaction column at 93.6%. Before dropping any of them I will check whether "missing" itself is linked to fraud.

### How many columns are almost empty, and how many are complete?

I filter `missing_pct` with a condition in square brackets and count what is left with `len`.

In [47]:
# Columns with more than 90% missing
print(len(missing_pct[missing_pct > 90]))

# Columns with no missing values
print(len(missing_pct[missing_pct == 0]))

12
20


**Result:** 12 columns are more than 90% empty, and only 20 of the 434 columns have no missing values. The other 414 columns all have gaps, which is normal for real payment data. Handling missing values will be a big part of preprocessing.

**Next question:** is a missing value itself a clue? I will compare the fraud rate for transactions with and without device data.

## 8. Is missing device data a clue?

I split the transactions into two groups, with device data and without it (based on whether `DeviceType` is empty), and compare their fraud rates.

A mask is a column of True/False values. Putting it inside square brackets keeps only the rows where it is True.

In [51]:
# Identify rows where DeviceType is present
device_mask = df_merged['DeviceType'].notna()

# Keep those transaction rows
with_device = df_merged[device_mask]

# Calculate their fraud percentage
print(with_device['isFraud'].mean() * 100)

7.961792486329096


**Result:** transactions **with** device data have a fraud rate of about 7.96%, more than double the overall 3.5%. So whether device data is present is itself linked to fraud, and I should not simply throw that information away.

In [52]:
# Identify rows where DeviceType is missing
missing_device_mask = df_merged['DeviceType'].isna()

# Keep those transaction rows
without_device = df_merged[missing_device_mask]

# Calculate their fraud percentage
print(without_device['isFraud'].mean() * 100)

2.1017054677250795


**Result:** transactions **without** device data have a fraud rate of about 2.10%.

| Group | Share of transactions | Fraud rate |
|---|---|---|
| With device data | 24% | 7.96% |
| Without device data | 76% | 2.10% |
| All | 100% | 3.5% |

The overall 3.5% is a mix of the two groups, so if one is above it the other has to be below. Transactions with device data are about 4 times more likely to be fraud (7.96 / 2.10).

**Finding 1:** whether device data is missing is a fraud signal on its own. When I fill missing values for Logistic Regression, I will add a 0/1 "was missing" flag so the model keeps this information. XGBoost can handle missing values directly.

## Where I stopped (Day 2)

Done: missing values, column counts, fraud rate with and without device data.

Next: `TransactionAmt` (min, max, average, fraud vs genuine), then fraud rate by `ProductCD`, card type, email domain and `DeviceType`, fraud over time, and 5 written findings.